# Benign package fetcher + feature extractor (PyPI + npm)
Stage 1: download the latest release of every seed from the registry (threads, resumable).
Stage 2: static feature extraction from the archives (multiprocessing, resumable) -> features.csv

Nothing is ever installed, imported or executed. Archives are read in memory only.

In [ ]:
%pip -q install rapidfuzz tqdm requests pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 25.8 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import shutil
import os

# Define the current and desired work directories
current_work_dir = '/content/work'
new_drive_work_dir = '/content/drive/MyDrive/SHIELD_DATA/a_new_folder'

# Create the new directory in Google Drive if it doesn't exist
os.makedirs(new_drive_work_dir, exist_ok=True)

# Copy contents from the current work directory to the new Google Drive directory
if os.path.exists(current_work_dir):
    for item in os.listdir(current_work_dir):
        s = os.path.join(current_work_dir, item)
        d = os.path.join(new_drive_work_dir, item)
        if os.path.isdir(s):
            shutil.copytree(s, d, symlinks=True, ignore_dangling_symlinks=True, dirs_exist_ok=True)
        else:
            shutil.copy2(s, d)
    print(f"Copied contents from {current_work_dir} to {new_drive_work_dir}")
else:
    print(f"Current work directory {current_work_dir} does not exist. No files to copy.")

Copied contents from /content/work to /content/drive/MyDrive/SHIELD_DATA/a_new_folder


The files that were already generated in the default temporary folder have been copied to your Google Drive. Now, I will modify the configuration to ensure future files are saved directly to your Drive.

In [ ]:
import os, io, re, json, csv, math, time, tarfile, zipfile, hashlib, random, warnings, ast
import urllib.parse, multiprocessing as mp
from collections import Counter
from concurrent.futures import ThreadPoolExecutor, as_completed

import requests, pandas as pd
from tqdm.auto import tqdm
from rapidfuzz import process
from rapidfuzz.distance import DamerauLevenshtein

# ---------------- config ----------------
WORK = "/content/work"                 # point this at /content/drive/MyDrive/... to persist across sessions
SEED_FILES = {"pypi": "pypi_benign_seeds.json", "npm": "npm_benign_seeds.json"}
LABEL = 0                              # 0 = benign
MAX_DOWNLOAD_MB = 50                   # archives bigger than this are logged as too_large (not silently dropped)
N_THREADS = 8
UA = "pkg-dataset-research/0.1 (contact: you@example.com)"   # <- put a real contact
TOP_POP_PYPI = 5000
TOP_POP_NPM = 5000

for d in ("cache/pypi", "cache/npm"):
    os.makedirs(f"{WORK}/{d}", exist_ok=True)

In [ ]:
# Upload the two seed files (skipped if they're already in the working dir)
try:
    from google.colab import files
    if not all(os.path.exists(f) for f in SEED_FILES.values()):
        files.upload()
except ImportError:
    pass
SEEDS = {eco: sorted(set(json.load(open(f)))) for eco, f in SEED_FILES.items()}
print({k: len(v) for k, v in SEEDS.items()})

Saving pypi_benign_seeds.json to pypi_benign_seeds.json
Saving npm_benign_seeds.json to npm_benign_seeds.json
{'pypi': 3500, 'npm': 1691}


## Stage 1 - fetch from the registries

In [ ]:
S = requests.Session()
S.headers["User-Agent"] = UA

class TooLarge(Exception): pass

def get_json(url, tries=4):
    for i in range(tries):
        try:
            r = S.get(url, timeout=30)
            if r.status_code == 404:
                return None, 404
            if r.status_code in (429, 500, 502, 503, 504):
                time.sleep(2 ** i + random.random()); continue
            r.raise_for_status()
            return r.json(), 200
        except (requests.RequestException, ValueError):
            time.sleep(2 ** i)
    return None, -1

def download(url, dest, expect_hex, algo, max_mb=MAX_DOWNLOAD_MB, tries=3):
    if os.path.exists(dest):
        return "cached"
    tmp = dest + ".part"
    for i in range(tries):
        h, size = hashlib.new(algo), 0
        try:
            with S.get(url, stream=True, timeout=60) as r:
                r.raise_for_status()
                with open(tmp, "wb") as f:
                    for chunk in r.iter_content(1 << 16):
                        size += len(chunk)
                        if size > max_mb * 2 ** 20:
                            raise TooLarge()
                        h.update(chunk); f.write(chunk)
            if expect_hex and h.hexdigest() != expect_hex:
                os.remove(tmp); return "hash_mismatch"
            os.replace(tmp, dest); return "ok"
        except TooLarge:
            if os.path.exists(tmp): os.remove(tmp)
            return "too_large"
        except requests.RequestException:
            time.sleep(2 ** i)
    if os.path.exists(tmp): os.remove(tmp)
    return "download_failed"

def fetch_pypi(name):
    j, code = get_json(f"https://pypi.org/pypi/{urllib.parse.quote(name)}/json")
    if j is None:
        return {"name": name, "status": "not_found" if code == 404 else "api_error"}
    info, files_ = j["info"], j["urls"]
    pick = (next((f for f in files_ if f["packagetype"] == "sdist"), None)      # sdist first: it contains setup.py
            or next((f for f in files_ if f["packagetype"] == "bdist_wheel"), None))
    if not pick:
        return {"name": name, "status": "no_files", "version": info["version"]}
    dest = f"{WORK}/cache/pypi/{pick['filename']}"
    st = download(pick["url"], dest, pick["digests"].get("sha256"), "sha256")
    urls = list((info.get("project_urls") or {}).values()) + [info.get("home_page") or ""]
    reqs = [r for r in (info.get("requires_dist") or []) if "extra ==" not in r]
    cls = info.get("classifiers") or []
    return {"name": name, "status": st, "version": info["version"], "path": dest,
            "artifact_type": pick["packagetype"],
            "has_license": bool(info.get("license") or info.get("license_expression")
                                or any(c.startswith("License ::") for c in cls)),
            "has_repo_url": any(re.search(r"github|gitlab|bitbucket|codeberg|sr\.ht", u or "", re.I) for u in urls),
            "deps": [m.group(0) for r in reqs if (m := re.match(r"[A-Za-z0-9._-]+", r))]}

def fetch_npm(name):
    j, code = get_json("https://registry.npmjs.org/" + urllib.parse.quote(name, safe="@") + "/latest")
    if j is None:
        return {"name": name, "status": "not_found" if code == 404 else "api_error"}
    dist = j.get("dist", {})
    dest = f"{WORK}/cache/npm/{name.replace('/', '__')}-{j['version']}.tgz"
    st = download(dist["tarball"], dest, dist.get("shasum"), "sha1")
    repo = j.get("repository")
    repo = repo.get("url", "") if isinstance(repo, dict) else (repo or "")
    return {"name": name, "status": st, "version": j["version"], "path": dest, "artifact_type": "npm_tgz",
            "has_license": bool(j.get("license") or j.get("licenses")),
            "has_repo_url": bool(repo or j.get("homepage")),
            "deps": list((j.get("dependencies") or {}).keys())}

FETCHERS = {"pypi": fetch_pypi, "npm": fetch_npm}

def safe_fetch(eco, name):
    try:
        rec = FETCHERS[eco](name)
    except Exception as e:                       # never let one package kill the run
        rec = {"name": name, "status": f"exception:{type(e).__name__}"}
    rec["ecosystem"] = eco
    return rec

def run_fetch(eco):
    manifest = f"{WORK}/manifest_{eco}.jsonl"
    done = set()
    if os.path.exists(manifest):
        done = {json.loads(l)["name"] for l in open(manifest)}
    todo = [n for n in SEEDS[eco] if n not in done]
    with ThreadPoolExecutor(N_THREADS) as ex, open(manifest, "a") as out:
        futs = [ex.submit(safe_fetch, eco, n) for n in todo]
        for f in tqdm(as_completed(futs), total=len(futs), desc=f"fetch {eco}"):
            out.write(json.dumps(f.result()) + "\n"); out.flush()
    recs = [json.loads(l) for l in open(manifest)]
    print(eco, Counter(r["status"] for r in recs))
    return recs

MANIFEST = {eco: run_fetch(eco) for eco in ("pypi", "npm")}

fetch pypi:   0%|          | 0/3500 [00:00<?, ?it/s]

pypi Counter({'ok': 3440, 'too_large': 60})


fetch npm:   0%|          | 0/1691 [00:00<?, ?it/s]

npm Counter({'ok': 1681, 'too_large': 8, 'not_found': 1, 'download_failed': 1})


## Popular-package lists (for the name/typosquat features)

In [ ]:
def norm(name, eco):
    name = name.lower()
    return re.sub(r"[-_.]+", "-", name) if eco == "pypi" else name

def load_popular_pypi(n=TOP_POP_PYPI):
    for u in ("https://hugovk.github.io/top-pypi-packages/top-pypi-packages.min.json",
              "https://hugovk.github.io/top-pypi-packages/top-pypi-packages-30-days.min.json"):
        j, _ = get_json(u)
        if j:
            rows = sorted(j["rows"], key=lambda r: -r["download_count"])
            return [norm(r["project"], "pypi") for r in rows[:n]]
    raise RuntimeError("could not load top-pypi list")

def load_popular_npm(n=TOP_POP_NPM):
    names = []
    for frm in range(0, n, 250):
        j, _ = get_json("https://registry.npmjs.org/-/v1/search?text=not:deprecated"
                        f"&popularity=1.0&quality=0&maintenance=0&size=250&from={frm}")
        if not j or not j.get("objects"): break
        names += [o["package"]["name"].lower() for o in j["objects"]]
    print("npm popular list size:", len(names))
    return names

POP = {"pypi": load_popular_pypi(), "npm": load_popular_npm()}
POP_SET = {e: set(v) for e, v in POP.items()}
json.dump(POP, open(f"{WORK}/popular.json", "w"))

npm popular list size: 1


## Stage 2 - feature extraction

In [ ]:
CODE_EXT = {".py", ".pyw", ".js", ".mjs", ".cjs", ".ts", ".jsx", ".tsx", ".sh", ".bash", ".ps1", ".bat", ".cmd", ".pl", ".rb"}
BIN_EXT = {".exe", ".dll", ".so", ".dylib", ".bin", ".pyd", ".node", ".wasm", ".class", ".jar", ".o", ".a"}
MAGICS = (b"\x7fELF", b"MZ", b"\xca\xfe\xba\xbe", b"\xcf\xfa\xed\xfe", b"\x00asm")
READ_CAP = 2 * 2 ** 20            # max bytes analysed per file
PKG_CAP = 40 * 2 ** 20            # max bytes analysed per package
MAX_MEMBERS = 30000

def _rx(p): return re.compile(p)
RX = {
 "process_exec": _rx(r"\bos\.(?:system|popen|exec[lv]p?e?|spawn[lv]p?e?)\s*\(|\bsubprocess\.|\bpty\.spawn|\bcommands\.getoutput|\bchild_process\b|\b(?:execSync|spawnSync|execFile|execFileSync)\s*\(|\bshelljs\b|\bWScript\.Shell"),
 "network": _rx(r"\brequests\.(?:get|post|put|request|Session)|\burllib(?:2|\.request)?\b|\burlopen\s*\(|\bhttp\.client\b|\bhttplib\b|\bsocket\.socket|\bhttpx\.|\baiohttp\b|\bpycurl\b|require\(\s*['\"](?:node:)?(?:https?|net|dgram|dns|tls|node-fetch|axios|got|request)['\"]\s*\)|from\s+['\"](?:node:)?(?:https?|net|axios|node-fetch)['\"]|\bfetch\s*\(|\bXMLHttpRequest\b|\bnew\s+WebSocket\b|\bcurl\s|\bwget\s"),
 "dynamic_eval": _rx(r"(?<![\w.])eval\s*\(|(?<![\w.])exec\s*\(|__import__\s*\(|\bnew\s+Function\s*\(|\bvm\.(?:runIn\w+|Script)"),
 "decode": _rx(r"\bbase64\.(?:b64decode|decodebytes|urlsafe_b64decode|b32decode|a85decode)|\bbinascii\.(?:unhexlify|a2b_\w+)|\bbytes\.fromhex|\bcodecs\.decode|\.decode\(\s*['\"](?:hex|base64|rot13|zlib)|\batob\s*\(|Buffer\.from\([^)]*['\"](?:base64|hex)['\"]"),
 "fs_write": _rx(r"\bopen\s*\([^)]*['\"][wax]b?\+?['\"]|\bshutil\.(?:copy\w*|move)|\bos\.(?:makedirs|rename|chmod|remove|unlink)|\.write_(?:text|bytes)\s*\(|\bfs\.(?:writeFile|writeFileSync|appendFile|appendFileSync|createWriteStream|chmod\w*|copyFile\w*|rename\w*|mkdir\w*)"),
 "env_access": _rx(r"\bos\.environ|\bos\.getenv|\bprocess\.env\b"),
 "sensitive_paths": _rx(r"\.ssh\b|id_rsa|authorized_keys|\.aws\b|\.npmrc|\.pypirc|\.netrc|\.kube\b|\.docker/config|\.gnupg|\.git-credentials|/etc/passwd|/etc/shadow|Login Data|Local State|Web Data|Network/Cookies|cookies\.sqlite|\bkeychain|\bwallet(?:\.dat)?\b|metamask|\.bash_history|Local Storage/leveldb|google-chrome|BraveSoftware|Mozilla/Firefox|logins\.json"),
 "persistence": _rx(r"\bcrontab\b|/etc/cron|\bsystemctl\b|/etc/systemd|\.bashrc|\.zshrc|\.bash_profile|\.profile\b|LaunchAgents|LaunchDaemons|CurrentVersion\\+Run|\bschtasks\b|rc\.local|\bwinreg\b|Startup|/etc/init\.d|autostart"),
 # obfuscation counters (group D)
 "chr_getattr_concat": _rx(r"(?<![\w.])chr\s*\(|(?<![\w.])getattr\s*\(|\\x[0-9a-fA-F]{2}|\\u[0-9a-fA-F]{4}|\bfromCharCode\b|['\"]\s*\+\s*['\"]"),
 "marshal_zlib_pickle_compile": _rx(r"\bmarshal\b|\bzlib\b|\bpickle\b|\bcPickle\b|\blzma\b|\bbz2\b|(?<![\w.])compile\s*\(|\btypes\.CodeType"),
}
RISKY = ("process_exec", "network", "dynamic_eval", "decode", "sensitive_paths", "persistence")
URL_RX = _rx(r"https?://((?:[A-Za-z0-9-]+\.)+[A-Za-z]{2,}|\d{1,3}(?:\.\d{1,3}){3})(?::\d+)?")
IP_RX = _rx(r"(?<![\w.])(\d{1,3})\.(\d{1,3})\.(\d{1,3})\.(\d{1,3})(?![\w.]*\w)")
SUSP_DOMAIN_RX = _rx(r"discord(?:app)?\.com/api/webhooks|pastebin\.com|paste\.ee|hastebin|ghostbin|ngrok(?:-free)?\.(?:io|app|dev)|serveo\.net|localtunnel|loca\.lt|trycloudflare\.com|pagekite|requestbin|webhook\.site|pipedream\.net|burpcollaborator|oast\.|interact\.sh|transfer\.sh|api\.telegram\.org|hooks\.slack\.com|iplogger|grabify|canarytokens|bit\.ly|tinyurl|duckdns|ddns\.net|000webhostapp|glitch\.me|workers\.dev|repl\.co|\.onion")
STR_RX = _rx(r"\"([^\"\\\n]{20,2000})\"|'([^'\\\n]{20,2000})'")
B64_RX = _rx(r"[A-Za-z0-9+/]{40,}={0,2}")
HEX_RX = _rx(r"(?:[0-9a-fA-F]{2}){20,}")
ID_RX = _rx(r"[A-Za-z_$][A-Za-z0-9_$]*")
PY_HOOK_RX = _rx(r"\bcmdclass\s*=|class\s+\w+\(\s*(?:\w+\.)?(?:install|develop|egg_info|build_py|build_ext|bdist_wheel|sdist)\s*\)")
SHELLISH_RX = _rx(r"\b(?:node|sh|bash|zsh|cmd|powershell|pwsh|curl|wget|python3?|perl|ruby|npx|eval)\b|\.(?:c?js|mjs|sh|py|ps1|bat|cmd)\b")
HOOK_FILE_RX = _rx(r"[\w@./\\-]+\.(?:c?js|mjs|sh|py|ps1|bat|cmd)")

def entropy(s):
    n = len(s); return -sum(c / n * math.log2(c / n) for c in Counter(s).values())

def iter_archive(path):
    """Yield (relpath, declared_size, head_bytes). Reads in memory; nothing touches disk."""
    n = 0
    if path.endswith((".whl", ".zip")):
        with zipfile.ZipFile(path) as z:
            for zi in z.infolist():
                if zi.is_dir(): continue
                n += 1
                if n > MAX_MEMBERS: return
                with z.open(zi) as fh: yield zi.filename, zi.file_size, fh.read(READ_CAP)
    else:
        with tarfile.open(path, "r:*") as tf:
            for m in tf:
                if not m.isfile(): continue          # skips symlinks/devices
                n += 1
                if n > MAX_MEMBERS: return
                rel = m.name.split("/", 1)[1] if "/" in m.name else m.name   # strip top dir (package/, name-1.0/)
                fh = tf.extractfile(m)
                yield rel, m.size, (fh.read(READ_CAP) if fh else b"")

def scan_text(text):
    out = {k: sum(1 for _ in rx.finditer(text)) for k, rx in RX.items()}
    lines = text.split("\n")
    out["loc"] = sum(1 for l in lines if l.strip())
    out["max_line"] = max((len(l) for l in lines), default=0)
    out["nlines"] = len(lines)
    mx = 0.0
    for i, m in enumerate(STR_RX.finditer(text)):
        if i >= 2000: break
        mx = max(mx, entropy(m.group(1) or m.group(2)))
    out["max_ent"] = mx
    blob = 0
    for i, m in enumerate(B64_RX.finditer(text)):
        if i >= 500: break
        s = m.group(0)
        if len(set(s)) >= 12 and re.search(r"\d", s) and re.search(r"[a-zA-Z]", s): blob = max(blob, len(s))
    for i, m in enumerate(HEX_RX.finditer(text)):
        if i >= 500: break
        blob = max(blob, len(m.group(0)))
    out["max_blob"] = blob
    ids = ID_RX.findall(text[:300_000])
    out["n_ids"], out["n_single_ids"] = len(ids), sum(1 for t in ids if len(t) == 1)
    out["domains"] = set(m.group(1).lower() for m in URL_RX.finditer(text))
    out["urls"] = sum(1 for _ in URL_RX.finditer(text))
    out["ips"] = sum(1 for m in IP_RX.finditer(text)
                     if all(int(g) <= 255 for g in m.groups()) and m.group(1) not in ("0", "127"))
    out["susp_dom"] = sum(1 for _ in SUSP_DOMAIN_RX.finditer(text))
    return out

COMBO = {"utils", "util", "dev", "js", "py", "python", "node", "lib", "libs", "tools", "tool", "helper", "helpers",
         "sdk", "api", "client", "core", "cli", "official", "secure", "plus", "pro", "new", "fix", "2", "3", "ng", "next", "v2"}
HOMO = str.maketrans({"0": "o", "1": "l", "5": "s", "3": "e"})

def name_features(name, eco):
    n = norm(name, eco); pop = POP[eco]
    hits = process.extract(n, pop, scorer=DamerauLevenshtein.distance, limit=3)
    best = next(((c, d, i) for c, d, i in hits if c != n), None)      # exclude the package itself
    combo = homo = 0
    for p in pop[:3000]:
        if p != n and len(p) >= 4 and p in n and n.replace(p, "", 1).strip("-_.@/") in COMBO:
            combo = 1; break
    if any(ord(ch) > 127 for ch in name): homo = 1
    alt = n.translate(HOMO).replace("rn", "m").replace("vv", "w")
    if alt != n and alt in POP_SET[eco]: homo = 1
    if best and best[1] == 1 and sorted(best[0]) == sorted(n): homo = 1        # adjacent transposition
    return {"name_len": len(name),
            "min_edit_dist": best[1] if best else -1,
            "nearest_popular_rank": (best[2] + 1) if best else -1,
            "is_combosquat": combo, "has_homoglyph_or_swap": homo}

def extract_features(rec):
    eco, name, path = rec["ecosystem"], rec["name"], rec["path"]
    code, domains, analysed = {}, set(), 0
    n_bin = n_files = n_code = n_min = n_py = n_py_fail = max_depth = 0
    pkg_json, setup_py = None, None
    for rel, size, head in iter_archive(path):
        n_files += 1
        max_depth = max(max_depth, rel.count("/") + 1)
        if "node_modules/" in rel: continue
        ext = os.path.splitext(rel)[1].lower()
        if rel == "package.json": pkg_json = head
        if ext in CODE_EXT:
            if analysed > PKG_CAP: continue
            analysed += len(head)
            text = head.decode("utf-8", "ignore")
            f = scan_text(text)
            n_code += 1; code[rel] = f
            if rel == "setup.py": setup_py = text
            if (size > 2048 and f["nlines"] and len(text) / f["nlines"] > 200) or f["max_line"] > 2000 or rel.endswith(".min.js"):
                n_min += 1
            if ext in (".py", ".pyw") and size <= READ_CAP:
                n_py += 1
                try:
                    with warnings.catch_warnings():
                        warnings.simplefilter("ignore"); ast.parse(text)
                except Exception:
                    n_py_fail += 1
        elif ext in BIN_EXT or (head.startswith(MAGICS) and b"\0" in head[:4096]):
            n_bin += 1

    # ---- A. install hook
    hook_types, hook_files, runs_shell = [], set(), 0
    if setup_py and PY_HOOK_RX.search(setup_py):
        hook_types.append("setup_cmdclass"); hook_files.add("setup.py")
        runs_shell = int(code["setup.py"]["process_exec"] > 0)
    if pkg_json:
        try:
            scripts = json.loads(pkg_json.decode("utf-8", "ignore")).get("scripts") or {}
        except Exception:
            scripts = {}
        for k in ("preinstall", "install", "postinstall"):
            cmd = scripts.get(k)
            if isinstance(cmd, str):
                hook_types.append(f"npm_{k}")
                if SHELLISH_RX.search(cmd.replace("node-gyp", "")): runs_shell = 1
                for p in HOOK_FILE_RX.findall(cmd):
                    p = os.path.normpath(p.lstrip("./")).replace("\\", "/")
                    if p in code: hook_files.add(p)
    risky = lambda f: sum(f[k] for k in RISKY)
    in_hook = sum(risky(code[p]) for p in hook_files)
    total_risky = sum(risky(f) for f in code.values())

    # ---- aggregates
    S_ = lambda k: sum(f[k] for f in code.values())
    loc = S_("loc"); kloc = max(loc, 1) / 1000
    row = {"name": name, "ecosystem": eco, "label": LABEL, "version": rec["version"], "artifact_type": rec["artifact_type"],
           "has_install_hook": int(bool(hook_types)), "hook_type": "+".join(hook_types) or "none",
           "hook_runs_shell_or_node": runs_shell, "hook_file_loc": sum(code[p]["loc"] for p in hook_files),
           "n_risky_calls_in_hook_files": in_hook, "n_risky_calls_elsewhere": total_risky - in_hook}
    cats = {"n_process_exec": "process_exec", "n_network": "network", "n_dynamic_eval": "dynamic_eval", "n_decode": "decode",
            "n_fs_write": "fs_write", "n_env_access": "env_access", "n_sensitive_paths": "sensitive_paths", "n_persistence": "persistence"}
    for col, k in cats.items():
        row[col] = S_(k); row[col + "_per_kloc"] = round(S_(k) / kloc, 4)
    row["loc_total"] = loc
    # ---- C. chains (file level)
    row["chain_decode_exec"] = int(any(f["decode"] and (f["dynamic_eval"] or f["process_exec"]) for f in code.values()))
    row["chain_download_exec"] = int(any(f["network"] and (f["process_exec"] or f["dynamic_eval"]) for f in code.values()))
    row["chain_env_network"] = int(any(f["env_access"] and f["network"] for f in code.values()))
    # ---- D. obfuscation
    n_ids = S_("n_ids")
    row.update({"max_string_entropy": round(max((f["max_ent"] for f in code.values()), default=0), 4),
                "max_b64_or_hex_blob_len": max((f["max_blob"] for f in code.values()), default=0),
                "max_line_len": max((f["max_line"] for f in code.values()), default=0),
                "single_char_ident_ratio": round(S_("n_single_ids") / n_ids, 4) if n_ids else 0,
                "n_chr_getattr_concat": S_("chr_getattr_concat"),
                "n_marshal_zlib_pickle_compile": S_("marshal_zlib_pickle_compile"),
                "is_minified_ratio": round(n_min / n_code, 4) if n_code else 0})
    # ---- E. network indicators
    for f in code.values(): domains |= f["domains"]
    row.update({"n_url_literals": S_("urls"), "n_ip_literals": S_("ips"),
                "n_suspicious_domains": S_("susp_dom"), "n_unique_domains": len(domains)})
    # ---- F. structure
    row.update({"archive_kb": round(os.path.getsize(path) / 1024, 1), "file_count": n_files, "code_file_count": n_code,
                "n_binary_blobs": n_bin, "max_depth": max_depth,
                "parse_fail_ratio": round(n_py_fail / n_py, 4) if n_py else 0})
    # ---- G. name
    row.update(name_features(name, eco))
    # ---- H. metadata
    ver = rec["version"]
    try: major = int(re.match(r"\d+", ver).group(0))
    except Exception: major = 0
    deps = [norm(d, eco) for d in rec.get("deps", [])]
    row.update({"has_license": int(rec["has_license"]), "has_repo_url": int(rec["has_repo_url"]),
                "n_declared_deps": len(deps), "n_deps_low_popularity": sum(1 for d in deps if d not in POP_SET[eco]),
                "version_looks_bumped": int(major >= 90 or ver.startswith("0.0."))})
    return row

def safe_extract(rec):
    try:
        return extract_features(rec)
    except Exception as e:
        return {"name": rec["name"], "ecosystem": rec["ecosystem"], "label": LABEL, "error": f"{type(e).__name__}: {e}"[:200]}

In [ ]:
FIELDS = None
OUT = f"{WORK}/features.csv"
ERR = f"{WORK}/feature_errors.csv"

def run_features():
    global FIELDS
    recs = [r for eco in MANIFEST for r in MANIFEST[eco] if r["status"] in ("ok", "cached")]
    done = set()
    if os.path.exists(OUT):
        prev = pd.read_csv(OUT, usecols=["ecosystem", "name"]); done = set(zip(prev.ecosystem, prev.name))
    todo = [r for r in recs if (r["ecosystem"], r["name"]) not in done]
    print(f"{len(recs)} archives available, {len(todo)} to process")
    errors = []
    with mp.get_context("fork").Pool(os.cpu_count()) as pool:      # fork -> workers inherit POP / POP_SET
        for row in tqdm(pool.imap_unordered(safe_extract, todo, chunksize=4), total=len(todo)):
            if "error" in row:
                errors.append(row); continue
            new = not os.path.exists(OUT)
            if FIELDS is None:
                FIELDS = list(row.keys()) if new else list(pd.read_csv(OUT, nrows=0).columns)
            with open(OUT, "a", newline="") as f:
                w = csv.DictWriter(f, fieldnames=FIELDS, extrasaction="ignore", restval="")
                if new: w.writeheader()
                w.writerow(row)
    if errors: pd.DataFrame(errors).to_csv(ERR, index=False); print(len(errors), "extraction errors ->", ERR)

run_features()
df = pd.read_csv(OUT)
print(df.shape); print(df.groupby("ecosystem").size())
df.head()

5121 archives available, 5121 to process


  0%|          | 0/5121 [00:00<?, ?it/s]

(5121, 58)
ecosystem
npm     1681
pypi    3440
dtype: int64


,name,ecosystem,label,version,artifact_type,has_install_hook,hook_type,hook_runs_shell_or_node,hook_file_loc,n_risky_calls_in_hook_files,...,name_len,min_edit_dist,nearest_popular_rank,is_combosquat,has_homoglyph_or_swap,has_license,has_repo_url,n_declared_deps,n_deps_low_popularity,version_looks_bumped
0,a2wsgi,pypi,0,1.10.10,sdist,0,none,0,0,0,...,6,2,3277,0,0,1,1,1,0,0
1,about-time,pypi,0,4.2.2,sdist,0,none,0,0,0,...,10,5,591,0,0,1,1,0,0,0
2,absl-py,pypi,0,2.5.0,sdist,0,none,0,0,0,...,7,3,2011,0,0,1,1,0,0,0
3,acme,pypi,0,5.8.0,sdist,0,none,0,0,0,...,4,2,896,0,0,1,1,5,0,0
4,adbc-driver-manager,pypi,0,1.12.0,sdist,0,none,0,0,0,...,19,4,1379,0,0,1,1,1,0,0


In [ ]:
# Fetch-stage audit: which seeds did NOT make it into features.csv, and why
audit = pd.DataFrame([r for eco in MANIFEST for r in MANIFEST[eco]])
print(audit.groupby(["ecosystem", "status"]).size())
try:
    from google.colab import files; files.download(OUT)
except ImportError:
    pass

ecosystem  status         
npm        download_failed       1
           not_found             1
           ok                 1681
           too_large             8
pypi       ok                 3440
           too_large            60
dtype: int64


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import json

# Assuming WORK is defined as in previous cells
# If WORK is not defined, it should be defined as WORK = "/content/work"

with open(f"{WORK}/popular.json", "r") as f:
    popular_data = json.load(f)

print("Popular PyPI packages (first 5):")
print(popular_data['pypi'][:5])
print("\nPopular npm packages (first 5):")
print(popular_data['npm'][:5])

In [ ]:
# Colab: run as one cell. Upload your current popular.json when prompted.
# Keeps the existing "pypi" list untouched and REPLACES "npm" with a ranked list of popular npm packages.
import io, os, re, json, time, random, tarfile
import requests

UA = "pkg-dataset-research/0.1 (contact: you@example.com)"   # <- put a real contact
TARGET = 5000                                                   # npm names to keep (same as pypi)
S = requests.Session(); S.headers["User-Agent"] = UA

NAME_RE = re.compile(r"^(?:@[a-z0-9][a-z0-9._~-]*/)?[a-z0-9][a-z0-9._~-]*$")
QUOTED_RE = re.compile(r"""["']((?:@[^"'\s/]+/)?[^"'\s/@]+)["']""")

def get(url, tries=4):
    for i in range(tries):
        try:
            r = S.get(url, timeout=60)
            if r.status_code in (429, 500, 502, 503, 504):
                time.sleep(2 ** i + random.random()); continue
            return r if r.status_code == 200 else None
        except requests.RequestException:
            time.sleep(2 ** i)
    return None

def dedupe(names):
    seen, out = set(), []
    for n in names:
        n = n.strip().lower()
        if n and n not in seen and NAME_RE.match(n):
            seen.add(n); out.append(n)
    return out

# ---- Source A: the `npm-high-impact` package ships a ranked list of the most downloaded / depended-on packages
def from_high_impact():
    meta = get("https://registry.npmjs.org/npm-high-impact/latest")
    if not meta: return []
    tgz = get(meta.json()["dist"]["tarball"])
    if not tgz: return []
    best = []
    with tarfile.open(fileobj=io.BytesIO(tgz.content), mode="r:gz") as tf:
        for m in tf:
            if m.isfile() and m.name.endswith((".js", ".mjs", ".cjs", ".json")):
                txt = tf.extractfile(m).read().decode("utf-8", "ignore")
                cand = [n for n in QUOTED_RE.findall(txt) if NAME_RE.match(n)]
                if len(cand) > len(best): best = cand          # the file holding the big array
    return dedupe(best)

# ---- Source B: registry search API sorted by popularity (several query variants; first that works wins)
SEARCH_VARIANTS = [
    "text=not:unstable&popularity=1.0&quality=0.0&maintenance=0.0",
    "text=not:deprecated&popularity=1.0",
    "text=not:insecure&popularity=1.0",
    "text=%20&popularity=1.0",
]
def from_search(target):
    for v in SEARCH_VARIANTS:
        first = get(f"https://registry.npmjs.org/-/v1/search?{v}&size=250&from=0")
        if not first or len(first.json().get("objects", [])) < 100:
            print("  search variant gave <100 results, skipping:", v); continue
        names = [o["package"]["name"] for o in first.json()["objects"]]
        for frm in range(250, target, 250):
            r = get(f"https://registry.npmjs.org/-/v1/search?{v}&size=250&from={frm}")
            if not r or not r.json().get("objects"): break
            names += [o["package"]["name"] for o in r.json()["objects"]]
        print("  search variant OK:", v, "->", len(names))
        return dedupe(names)
    return []

def build_npm_list(target=TARGET):
    a = from_high_impact(); print("npm-high-impact:", len(a))
    b = from_search(target) if len(a) < target else []; print("search API:", len(b))
    return dedupe(a + b)[:target]

def update_popular(path_in, path_out, target=TARGET):
    pop = json.load(open(path_in))
    print("before:", {k: len(v) for k, v in pop.items()})
    npm = build_npm_list(target)
    if len(npm) < 500:
        raise RuntimeError(f"only {len(npm)} npm names collected; not overwriting. Check network or paste a list manually.")
    pop["npm"] = npm
    json.dump(pop, open(path_out, "w"))
    print("after:", {k: len(v) for k, v in pop.items()}, "| first 15 npm:", npm[:15])
    return pop

# ---------------- run ----------------
try:
    from google.colab import files
    if not os.path.exists("popular.json"): files.upload()
except ImportError:
    files = None

if os.path.exists("popular.json"):
    os.replace("popular.json", "popular_backup.json")
    POP = update_popular("popular_backup.json", "popular.json")
    POP_SET = {e: set(v) for e, v in POP.items()}              # drop-in for the pipeline's globals
    if files: files.download("popular.json")

Saving popular.json to popular.json
before: {'pypi': 5000, 'npm': 1}
npm-high-impact: 17334
search API: 0
after: {'pypi': 5000, 'npm': 5000} | first 15 npm: ['semver', 'minimatch', 'debug', 'ansi-styles', 'brace-expansion', 'strip-ansi', 'ansi-regex', 'ms', 'supports-color', 'chalk', 'commander', 'lru-cache', 'string-width', 'wrap-ansi', 'tslib']


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>